# LFS Data Pull & Validation\n\nPull and validate **Statistics Canada Labour Force Survey** data via the Web Data Service (WDS) REST API (no API key). Source: **Table 14-10-0287-01** (product ID `14100287`) -- monthly unemployment / participation / employment rates by geography, gender, and age group.\n\nRun the cells top to bottom. The last data cell is a **known-answer check** you compare against the official StatCan figure -- the project's regression oracle.\n\n> **Governance caution (see `CLAUDE.md`):** the LFS measures labour-market *stress* only. It cannot establish individual eligibility, poverty, household need, disability, food insecurity, or housing insecurity. Treat every output as a claim to check, flag suppressed/small cells, and route any funding-relevant read to a human reviewer.\n\nRequires `pandas` and `matplotlib` (`pip install pandas matplotlib`).

## Setup

In [ ]:
import io, json, zipfile, urllib.request
from datetime import date

WDS = "https://www150.statcan.gc.ca/t1/wds/rest"   # StatCan Web Data Service (no API key)
PID = 14100287                                       # Table 14-10-0287-01 (LFS, monthly)
UA  = {"User-Agent": "MMA616-GovernedClaude/1.0 (educational project)"}

def _get(url, binary=False):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=60) as r:
        data = r.read()
    return data if binary else data.decode("utf-8")

def _post(path, payload):
    body = json.dumps(payload).encode()
    req = urllib.request.Request(f"{WDS}/{path}", data=body,
                                 headers={**UA, "Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read().decode())

print("Setup OK. Target table:", PID)

## 1. Freshness check\nOnly pull fresh data when the table actually changed.

In [ ]:
# Freshness gate: did the LFS table change today? (it updates ~monthly)
on = date.today().isoformat()
changed = json.loads(_get(f"{WDS}/getChangedCubeList/{on}")).get("object", [])
pids = {int(i["productId"]) for i in changed}
print(f"{len(pids)} StatCan tables changed on {on}.")
print("-> LFS table changed today; pull fresh data." if PID in pids
      else "-> No LFS change today. Most days are empty; that is expected.")

## 2. Explore dimensions and member codes

In [ ]:
# Dimensions + member codes. Use these IDs to build a `coordinate` for targeted pulls.
meta = _post("getCubeMetadata", [{"productId": PID}])[0]["object"]
print(meta["cubeTitleEn"], "| range:", meta["cubeStartDate"], "->", meta["cubeEndDate"], "\n")
for dim in meta["dimension"]:
    print(f"Dim {dim['dimensionPositionId']}: {dim['dimensionNameEn']}")
    for m in dim["member"][:12]:
        print(f"   [{m['memberId']:>3}] {m['memberNameEn']}")
    if len(dim["member"]) > 12:
        print(f"   ... ({len(dim['member'])} members total)")
    print()

## 3. Download the full table

In [ ]:
import pandas as pd   # if missing:  pip install pandas matplotlib

info = json.loads(_get(f"{WDS}/getFullTableDownloadCSV/{PID}/en"))
assert info.get("status") == "SUCCESS", info
print("Downloading:", info["object"])
blob = _get(info["object"], binary=True)

with zipfile.ZipFile(io.BytesIO(blob)) as zf:
    name = next(n for n in zf.namelist()
                if n.lower().endswith(".csv") and "metadata" not in n.lower())
    df = pd.read_csv(zf.open(name), low_memory=False)

# the characteristic column ("Labour force characteristics") and value column
char_col = next(c for c in df.columns if "characteristic" in c.lower())
df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce")
print("Downloaded:", df.shape)
df.head()

## 4. Headline-series filter\nA helper that isolates one comparable series (seasonally adjusted, all genders, 15+).

In [ ]:
def headline(frame):
    """Filter to the standard national/provincial headline series:
    seasonally adjusted, all genders, 15 years and over -> one value per period."""
    f = frame
    for col in f.columns:
        cl = col.lower()
        if "data type" in cl:
            f = f[f[col].astype(str).str.contains("Seasonally", case=False, na=False)]
        elif "gender" in cl or cl == "sex":
            f = f[f[col].astype(str).str.contains("Total", case=False, na=False)]
        elif "age group" in cl:
            f = f[f[col].astype(str).str.contains("15 years and over", case=False, na=False)]
    return f

print("headline() ready.")

## 5. Tidy slice -> CSV for the dashboard

In [ ]:
# Keep just the three headline rate indicators and save a tidy CSV for the dashboard.
rates = ["unemployment rate", "participation rate", "employment rate"]
tidy = df[df[char_col].str.lower().isin(rates)].copy()
tidy.to_csv("lfs_14100287_tidy.csv", index=False)
print("Tidy rows:", tidy.shape, "-> lfs_14100287_tidy.csv")
tidy[[c for c in ["REF_DATE","GEO",char_col,"VALUE"] if c in tidy.columns]].head()

## 6. Validate the data

In [ ]:
# Basic data-validation checks
print("Columns:", list(df.columns))
print("Total rows:", f"{len(df):,}")
print("Date range:", df["REF_DATE"].min(), "->", df["REF_DATE"].max())
print("Geographies:", df["GEO"].nunique(), "->", sorted(df["GEO"].unique()))
print("\nRows per indicator:")
print(df[char_col].value_counts())
print("\nMissing/blank VALUE cells:", int(df["VALUE"].isna().sum()),
      "(blanks are suppressed/unavailable cells -- flag, never invent)")

### Known-answer (regression-oracle) check

In [ ]:
# KNOWN-ANSWER / regression-oracle check.
# Pull the latest national unemployment rate (SA, all genders, 15+) and
# eyeball it against the official StatCan figure -- this is your evaluation anchor.
latest = df["REF_DATE"].max()
h = headline(df[(df["GEO"] == "Canada") &
                (df[char_col] == "Unemployment rate") &
                (df["REF_DATE"] == latest)])
print(f"Latest period: {latest}")
print("Canada unemployment rate (SA, 15+, all genders):", list(h["VALUE"]), "%")
print("\nVALIDATE against the official sources:")
print("  Table:    https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1410028701")
print("  The Daily: https://www150.statcan.gc.ca/n1/dai-quo/index-eng.htm  (Labour Force Survey)")
print("If these disagree, STOP and investigate before trusting any dashboard number.")

## 7. Quick visual sanity check

In [ ]:
import matplotlib.pyplot as plt

prov = "Alberta"   # try any province from the GEO list above
s = headline(df[(df["GEO"] == prov) & (df[char_col] == "Unemployment rate")])
s = s.sort_values("REF_DATE").tail(60)   # ~5 years of monthly data

plt.figure(figsize=(9, 4))
plt.plot(pd.to_datetime(s["REF_DATE"]), s["VALUE"], marker=".")
plt.title(f"{prov}: unemployment rate (seasonally adjusted, 15+) -- last 5 years")
plt.ylabel("%"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Next steps\n- Compare the oracle value to the official StatCan figure; record the match as your Day-2 evaluation baseline.\n- Pick the lean slice for the dashboard (e.g., provincial unemployment + participation + youth 15-24 + one industry cut + trend).\n- Re-run `1. Freshness check` on each LFS release day, then re-pull only if it changed (keeps the source live).\n- Anything that reads as *who needs assistance* rather than *where labour-market stress is high* belongs on the CANNOT list.